# WTI Crude Oil — Protected Evaluation (Notebook 6 of 7)

> **Part 6 of 7.** Requires Notebook 5 to have been run first —
> the adaptive agent strategy variants must be trained.

This is the culminating comparison: all stateless predictors from Notebook 4
versus all three trained adaptive agent variants (plus the untrained baseline)
on the **held-out 2026 data**.

The evaluation period is Feb–Mar 2026 — the heart of the Persian Gulf
geopolitical price shock. Neither the stateless methods nor the adaptive agent
has seen this data. The question is whether the agent's 2025 training improved
its calibration for exactly the kind of regime it was trained on.

| | Stateless methods | Untrained agent | Trained agent variants |
|---|---|---|---|
| Training | None | None | 2025 curriculum (NB05) |
| Eval data | 2026 (never seen) | 2026 (never seen) | 2026 (never seen) |
| Strategy updates during eval | N/A | **Frozen** | **Frozen** |

Three trained variants are compared — one per NB05 training activity:
`wti-strategy-act1` (self-directed), `wti-strategy-stats` (stats curriculum),
`wti-strategy-news` (news curriculum). The untrained agent provides the
adaptive agent's own baseline — isolating the value of training from the
value of having an adaptive strategy at all.

---
## 0. Setup & Freeze

In [13]:
import warnings
from pathlib import Path

import pandas as pd

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
)
from aieng.forecasting.evaluation.backtest import BacktestResult
from energy_oil_forecasting.adaptive_agent import build_wti_adaptive_predictor
from energy_oil_forecasting.adaptive_agent.curriculum.snapshot_utils import (
    state_checksum,
)
from energy_oil_forecasting.analysis import score_backtest_results
from energy_oil_forecasting.data import build_wti_service

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path('.')
_SKILLS_ROOT = _NB_DIR / 'adaptive_agent' / 'skills'
_CURRICULUM_DIR = _NB_DIR / 'adaptive_agent' / 'curriculum'
_SPECS_DIR = _NB_DIR / 'specs'

SEED_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy'       # untrained baseline
ACT1_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-act1'  # Act 1: self-directed
STATS_STRATEGY_DIR = _SKILLS_ROOT / 'wti-strategy-stats' # Act 2a: stats curriculum
NEWS_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-news'  # Act 2b: news curriculum

# All four variants evaluated in Section 3:
ADAPTIVE_VARIANTS = {
    'Agent — untrained':    SEED_STRATEGY_DIR,
    'Agent — Act 1':        ACT1_STRATEGY_DIR,
    'Agent — Act 2a (stats)': STATS_STRATEGY_DIR,
    'Agent — Act 2b (news)':  NEWS_STRATEGY_DIR,
}

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = 'gemini-3.5-flash'

# ── Run guard ─────────────────────────────────────────────────────────────────
RUN_EVAL = True   # Set True on first run; commit outputs; leave False.

# ── Data service ──────────────────────────────────────────────────────────────
data_service = build_wti_service()
print('Setup complete.')

Setup complete.


In [14]:
# ── Freeze: record pre-eval checksums for all adaptive variants ──────────────
# We verify post-eval that no skill state files were modified during eval.
_pre_eval_checksums = {name: state_checksum(d) for name, d in ADAPTIVE_VARIANTS.items()}
print('Pre-eval checksums recorded:')
for name, ck in _pre_eval_checksums.items():
    print(f'  {name}: {ck[:16]}...')

Pre-eval checksums recorded:
  Agent — untrained: 74abfb0fa9330f94...
  Agent — Act 1: f6c957f468a74eb7...
  Agent — Act 2a (stats): 04b1ee3e0a6dc840...
  Agent — Act 2b (news): 26c56f60fe602076...


---
## 1. The Knowledge-Cutoff Teaching Point

**Gemini's parametric knowledge cutoff is approximately January 2025.**
This has a concrete implication for this evaluation:

- The **training period** (2025) is at or beyond the model's parametric
  knowledge horizon. During curriculum delivery in NB05, the agent could not
  rely on memorized facts about 2025 WTI prices — it had to reason from the
  backtest report and pre-cached news summaries we provided.

- The **evaluation period** (Feb–Mar 2026) is definitively post-cutoff.
  During eval, the agent must rely entirely on:
  1. Its Google Search tool (with `cutoff_date` enforcement per origin)
  2. Its code execution tool (for statistical analysis of available data)
  3. Its accumulated strategy state (calibration corrections from training)

This is a clean test of what the training phase actually adds: it cannot be
attributed to the model's parametric knowledge of the eval period.

---
## 2. Load Stateless Eval Results

Notebook 4 saved the 2026 eval results for the top stateless predictors.
We load them here — no re-run needed.

In [15]:
# ── Load eval results from NB04 ─────────────────────────────────────────────
_eval_jsons = sorted(_CURRICULUM_DIR.glob('eval_*.json'))
if not _eval_jsons:
    raise FileNotFoundError(
        'No eval result files found in adaptive_agent/curriculum/. '
        'Run 04_systematic_backtest_eval.ipynb first.'
    )

all_eval_results: dict[str, BacktestResult] = {}
for f in _eval_jsons:
    name = f.stem.removeprefix('eval_')
    all_eval_results[name] = BacktestResult.model_validate_json(f.read_text())

print(f'Loaded {len(all_eval_results)} stateless eval result(s):')
for name, r in all_eval_results.items():
    print(f'  {name}: {len(r.predictions)} predictions, '
          f'mean CRPS = {r.mean_crps:.4f}')

Loaded 3 stateless eval result(s):
  AutoARIMA: 22 predictions, mean CRPS = 10.9984
  Naive (Last Value): 22 predictions, mean CRPS = 13.6432
  Prophet: 16 predictions, mean CRPS = 20.7483


---
## 3. Run Adaptive Agent Variants on Eval Spec

Each adaptive agent variant is evaluated on the same 2026 eval spec  
(`energy_oil_eval.yaml`) used by the stateless predictors in NB04.

> **Run guard:** `RUN_EVAL = False` by default. Set to `True` on first run,
> commit the saved result files, and leave `False` for reproducibility.

In [ ]:
import yaml  # noqa: PLC0415
with open(_SPECS_DIR / 'energy_oil_eval.yaml') as _f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(_f))

def _safe_key(name: str) -> str:
    return name.replace(' ', '_').replace('(', '').replace(')', '').replace('—', '').strip('_')

if RUN_EVAL:
    print('Running all adaptive agent variants on 2026 eval spec...')
    print('(Live API calls — first run may take several minutes.)\n')

    for variant_name, strategy_dir in ADAPTIVE_VARIANTS.items():
        predictor = build_wti_adaptive_predictor(strategy_dir=strategy_dir)
        result_dict = cached_multi_backtest(predictor, eval_spec, data_service)
        result = next(iter(result_dict.values()))
        all_eval_results[variant_name] = result
        safe = _safe_key(variant_name)
        (_CURRICULUM_DIR / f'eval_{safe}.json').write_text(
            result.model_dump_json(), encoding='utf-8'
        )
        print(f'  {variant_name}: mean CRPS = {result.mean_crps:.4f} ✓')

    print('\nEval complete.')
else:
    # Load committed adaptive eval results if present
    for variant_name in ADAPTIVE_VARIANTS:
        safe = _safe_key(variant_name)
        _f = _CURRICULUM_DIR / f'eval_{safe}.json'
        if _f.exists():
            all_eval_results[variant_name] = BacktestResult.model_validate_json(
                _f.read_text()
            )
    print('RUN_EVAL = False — using committed outputs (or set True to re-run).')
    print(f'Eval results available: {list(all_eval_results)}')

Running all adaptive agent variants on 2026 eval spec...
(Live API calls — first run may take several minutes.)



---
## 4. Comparative Scorecard

All predictors on the same 2026 eval origins.

In [ ]:
scorecard_rows = []
for name, result in all_eval_results.items():
    scores = score_backtest_results(result, data_service)
    scorecard_rows.append(
        {
            'Predictor': name,
            'Mean CRPS': scores.get('mean_crps', float('nan')),
            'MAE h=21d': scores.get('mae_h21', float('nan')),
            '80% Coverage': scores.get('coverage_80', float('nan')),
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index('Predictor')
df_scorecard = df_scorecard.sort_values('Mean CRPS')

print('━' * 72)
print('2026 PROTECTED EVAL — ALL PREDICTORS:')
print('━' * 72)
print(df_scorecard.to_string())

# Coverage vs. 80% target
print('\nCoverage vs. 80% target:')
for name, row in df_scorecard.iterrows():
    cov = row['80% Coverage']
    delta = cov - 0.80
    direction = 'over' if delta > 0 else 'under'
    print(f'  {name}: {cov:.1%} ({direction} by {abs(delta):.1%})')

---
## 5. Freeze Verification

Confirm that the evaluation did not trigger any skill state mutations.
The checksums should match the pre-eval values recorded in Setup.

In [ ]:
print('State integrity check (all variants should be unchanged):')
all_ok = True
for name, d in ADAPTIVE_VARIANTS.items():
    ck_after = state_checksum(d)
    ok = ck_after == _pre_eval_checksums[name]
    all_ok = all_ok and ok
    print(f'  {name}: {"✓ unchanged" if ok else "⚠ MODIFIED"}')

if not all_ok:
    print('\nWarning: at least one agent updated its strategy during evaluation.')
    print('See the closing note for how to explore this intentionally.')

---
## 6. Closing Note — Unfreezing

The adaptive agent evaluated here was **frozen**: its strategy state was not
updated during evaluation. This gives a clean before/after comparison between
trained and stateless predictors on identical eval origins.

But in live deployment, you would not freeze the agent. After each resolved
prediction, you would send a resolution message and let the agent decide whether
to record an observation or update a hypothesis. Over time, the strategy evolves.

**To explore unfreezing:**

1. Set `RUN_EVAL = True`.
2. Remove the state checksum assertion (or ignore the warning).
3. Modify the eval loop to send a resolution message after each prediction:

```python
# After each prediction resolves:
resolution_msg = (
    f'The actual WTI price on {pred.forecast_date.date()} was {actual:.2f}. '
    f'Your point forecast was {pred.payload.point_forecast:.2f} '
    f'(error: {pred.payload.point_forecast - actual:+.2f}). '
    'Please review whether this outcome is relevant to any open hypothesis.'
)
await runner.run_text_async(resolution_msg)
```

4. Re-run and compare the final strategy state to the frozen baseline.

Notebook 7 shows how to do this interactively via `adk web`.